# 数据获取及清洗

负责人：李业童

## 数据获取

从国家数据官网下载了所需的四个文件：

city_income.csv;
city_expenditure.csv;
gdp.csv;
individual_deposit.csv;

用python导出目录结构并导入到 Gemini 里，以便代码编写。


In [3]:
# 输出文件目录
import os

def print_tree(start_path, prefix=""):
    items = os.listdir(start_path)
    items.sort()  # 排序，输出更整齐
    
    for i, item in enumerate(items):
        path = os.path.join(start_path, item)
        is_last = i == len(items) - 1
        
        if is_last:
            print(prefix + "└── " + item)
            new_prefix = prefix + "    "
        else:
            print(prefix + "├── " + item)
            new_prefix = prefix + "│   "
        
        if os.path.isdir(path):
            print_tree(path, new_prefix)

# 使用
print_tree(".")

├── 01_Lyt.ipynb
├── 01_data_clean.ipynb
├── 02_data_analysis.ipynb
├── README.md
├── data_clean
├── data_raw
│   ├── city_expenditure.csv
│   ├── city_income.xlsx.csv
│   ├── gdp.csv
│   └── individual_deposit.csv
└── output


## 数据合并
使用 python 对下载的三个文件进行合并，并输出到output

In [7]:
import pandas as pd
import os

print(f"当前工作路径: {os.getcwd()}")
print("-" * 30)

def load_and_clean_nbs_data(filepath, value_name):
    """
    专门针对国家统计局下载的 CSV 格式的清洗函数
    """
    print(f"正在读取并清洗: {filepath}")
    
    # 1. 明确跳过前 3 行 (skiprows=3)，直接读取第 4 行作为表头
    # 尝试 gbk 和 utf-8 编码以防报错
    try:
        df = pd.read_csv(filepath, encoding='gbk', skiprows=3)
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding='utf-8', skiprows=3)
        
    # 2. 将第一列（通常叫“地区”）重命名为 city
    first_col = df.columns[0]
    df.rename(columns={first_col: 'city'}, inplace=True)
    
    # 3. 宽表转长表
    df_melt = df.melt(id_vars=['city'], var_name='year_str', value_name=value_name)
    
    # 4. 清理无效的“城市”行（统计局经常在文件末尾加“注：...”）
    df_melt = df_melt.dropna(subset=['city'])
    df_melt = df_melt[~df_melt['city'].astype(str).str.contains('注|说明|来源', na=False)]
    df_melt['city'] = df_melt['city'].astype(str).str.strip()
    
    # 5. 清理年份：从 "2022年" 提取出 2022 并转为整数
    df_melt['year'] = df_melt['year_str'].astype(str).str.extract(r'(\d{4})')
    df_melt = df_melt.dropna(subset=['year']) # 删掉提取不出年份的脏数据
    df_melt['year'] = df_melt['year'].astype(int)
    
    # 6. 清理数值：强制转为数字，遇到空白或 "--" 会变成 NaN
    df_melt[value_name] = pd.to_numeric(df_melt[value_name], errors='coerce')
    
    # 返回干净的 3 列
    return df_melt[['city', 'year', value_name]]

# ==========================================
# 开始读取并合并四个文件
# ==========================================

# 1. 分别处理四个文件
df_income = load_and_clean_nbs_data('data_raw/city_income.xlsx.csv', 'income')
df_expend = load_and_clean_nbs_data('data_raw/city_expenditure.csv', 'expend')
df_gdp = load_and_clean_nbs_data('data_raw/gdp.csv', 'gdp')
df_deposit = load_and_clean_nbs_data('data_raw/individual_deposit.csv', 'deposit')

print("-" * 30)
print("正在合并数据集...")

# 2. 依次按 city 和 year 外连接（outer merge）合并
final_df = pd.merge(df_income, df_expend, on=['city', 'year'], how='outer')
final_df = pd.merge(final_df, df_gdp, on=['city', 'year'], how='outer')
final_df = pd.merge(final_df, df_deposit, on=['city', 'year'], how='outer')

# 3. 剔除那些由于表结构错位导致 4 个指标全是 NaN 的废行
final_df = final_df.dropna(how='all', subset=['income', 'expend', 'gdp', 'deposit'])

# 4. 按城市和年份排序一下，让数据更好看
final_df = final_df.sort_values(['city', 'year'], ascending=[True, False]).reset_index(drop=True)

# 5. 保存到 data_clean 文件夹
output_path = 'data_clean/merged_data.csv'
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 完美！合并后的干净数据已成功保存至: {output_path}")

# 预览前5行
final_df.head()

当前工作路径: c:\Users\EnderKi\Documents\GitHub\PA10_ex_Team01\city_finance_analysis
------------------------------
正在读取并清洗: data_raw/city_income.xlsx.csv
正在读取并清洗: data_raw/city_expenditure.csv
正在读取并清洗: data_raw/gdp.csv
正在读取并清洗: data_raw/individual_deposit.csv
------------------------------
正在合并数据集...
✅ 完美！合并后的干净数据已成功保存至: data_clean/merged_data.csv


,city,year,income,expend,gdp,deposit
0,上海,2024,8374.17,9874.84,53759.5,63910.99
1,上海,2023,8312.50,9638.51,51404.5,58319.93
2,上海,2022,7608.00,9393.00,48594.5,52638.00
3,上海,2021,7771.80,8430.86,47059.4,41150.40
4,上海,2020,7046.30,8102.11,41603.9,36733.97
